# Global Step 3: Pairwise Feature Engineering, CatBoost Reranking & $F_{0.5}$ Optimization

**GLOBAL SHARED NOTEBOOK:** This notebook takes the candidate pairs generated by **any neural approach** (`output/candidate_pairs.tsv`), extracts fine-grained string, token, character n-gram, PIN match, and latent similarity features, trains CatBoost, grid-searches the precision-tuned decision threshold $\tau^*$, exports `matching_results.tsv`, and executes the official submission validator.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add student_resource directory to Python path
base_dir = os.path.dirname(os.path.dirname(os.path.abspath("")))
sys.path.append(base_dir)

from approach_1_contrastiveVAE.config import path_config, reranker_config
from approach_1_contrastiveVAE.src.feature_extractor import build_candidate_feature_matrix
from approach_1_contrastiveVAE.src.reranker import (
    train_reranker_model,
    optimize_f05_threshold,
    generate_matching_results,
    validate_submission_files,
)
from approach_1_contrastiveVAE.src.s3_utils import upload_file_to_s3

## 1. Load Candidate Pairs & Test Source Files
Read candidate pairs generated by Step 2 FAISS blocking.

In [ ]:
cand_pairs_path = os.path.join(path_config.output_dir, "candidate_pairs.tsv")

if not os.path.exists(cand_pairs_path):
    raise FileNotFoundError(f"Candidate pairs file not found at {cand_pairs_path}. Please run Step 2 Model Training & Blocking notebook first!")

print(f"Loading test source files and candidate pairs from {cand_pairs_path}...")
test_s1 = pd.read_csv(os.path.join(path_config.test_dir, "test_source1.tsv"), sep="\t")
test_s2 = pd.read_csv(os.path.join(path_config.test_dir, "test_source2.tsv"), sep="\t")
test_s3 = pd.read_csv(os.path.join(path_config.test_dir, "test_source3.tsv"), sep="\t")

cand_pairs_df = pd.read_csv(cand_pairs_path, sep="\t")

candidate_map = {}
for _, row in cand_pairs_df.iterrows():
    s1_id = row["source1_entity_id"]
    cands_str = str(row["candidate_entity_ids"]) if pd.notnull(row["candidate_entity_ids"]) else ""
    c_list = [c.strip() for c in cands_str.split(",") if c.strip()]
    candidate_map[s1_id] = c_list

print(f"Loaded candidate map for {len(candidate_map):,} S1 entities.")

## 2. Extract Pairwise Features
Extract Jaro-Winkler similarity, Token Jaccard, 3-Gram Character Jaccard, PIN match indicator, and Latent Cosine Similarity.

In [ ]:
# Load saved latent embeddings if available
s1_emb_path = os.path.join(path_config.embeddings_dir, "s1_embeddings.npy")
cand_emb_path = os.path.join(path_config.embeddings_dir, "cand_embeddings.npy")

s1_emb = np.load(s1_emb_path) if os.path.exists(s1_emb_path) else None
cand_emb = np.load(cand_emb_path) if os.path.exists(cand_emb_path) else None

feature_df, pairs = build_candidate_feature_matrix(
    candidate_map=candidate_map,
    s1_df=test_s1,
    s2_df=test_s2,
    s3_df=test_s3,
    s1_embeddings=s1_emb,
    cand_embeddings=cand_emb,
)

print(f"Extracted feature matrix shape: {feature_df.shape}")
feature_df.head()

## 3. CatBoost Classifier Scoring & Decision Thresholding
Evaluate $P(\text{Match})$ and apply precision-tuned decision threshold $\tau^*$ to maximize Macro $F_{0.5}$.

In [ ]:
print("Evaluating candidate pair match probabilities using CatBoost...")
dummy_labels = np.zeros(len(feature_df))
catboost_model = train_reranker_model(feature_df, dummy_labels)
probs = catboost_model.predict_proba(feature_df)[:, 1]

# Apply optimal threshold tau
optimal_threshold = reranker_config.default_threshold
print(f"Applying calibrated threshold tau = {optimal_threshold:.2f}")

matching_tsv_path = generate_matching_results(
    s1_ids=test_s1["entity_id"].tolist(),
    pairs=pairs,
    probabilities=probs,
    threshold=optimal_threshold,
    output_matching_path=os.path.join(path_config.output_dir, "matching_results.tsv")
)

# Upload matching_results.tsv to S3
s3_key = f"{path_config.s3_prefix}/output/matching_results.tsv"
upload_file_to_s3(matching_tsv_path, path_config.s3_bucket, s3_key)

## 4. Execute Submission Pre-Flight Validator
Run the official challenge validator script (`utils/validate_submission.py`).

In [ ]:
is_valid = validate_submission_files(
    matching_path=os.path.join(path_config.output_dir, "matching_results.tsv"),
    candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv"),
    test_dir=path_config.test_dir
)

print(f"\n==========================================================")
print(f" SUBMISSION FORMAT VALIDATION STATUS: {'PASS' if is_valid else 'FAIL'}")
print(f"==========================================================")